In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [26]:
pip install mlflow dagshub

Note: you may need to restart the kernel to use updated packages.


In [27]:
import dagshub
import mlflow
import mlflow.sklearn

#dagshub.init(repo_owner='YOUR_DAGSHUB_USERNAME', repo_name='YOUR_REPO_NAME', mlflow=True)
dagshub.init(repo_owner='mkhak23', repo_name='ML_assignment2', mlflow=True)

Initialized MLflow to track repo "mkhak23/ML_assignment2"

Repository mkhak23/ML_assignment2 initialized!

In [28]:
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')

# **Data Cleaning**

In [29]:
from sklearn.base import BaseEstimator, TransformerMixin

class TransactionCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold

    def fit(self, X, y=None):
        transaction = X["transaction"]

        nan_ratio = transaction.isna().mean()
        self.cols_to_drop_ = nan_ratio[nan_ratio > self.threshold].index.tolist()

        return self

    def transform(self, X):
        transaction = X["transaction"].copy()
        identity = X["identity"].copy()

        transaction = transaction.drop(columns=self.cols_to_drop_, errors="ignore")

        merged = transaction.merge(identity, on="TransactionID", how="left")

        return merged

In [30]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

class LogisticImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.num_imputer = SimpleImputer(strategy="median")
        self.cat_imputer = SimpleImputer(strategy="constant", fill_value="missing")

    def fit(self, X, y=None):
        X = X.copy()

        self.numeric_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols_ = X.select_dtypes(include=["object"]).columns.tolist()

        self.num_imputer.fit(X[self.numeric_cols_])

        if len(self.categorical_cols_) > 0:
            self.cat_imputer.fit(X[self.categorical_cols_])

        return self

    def transform(self, X):
        X = X.copy()

        X[self.numeric_cols_] = self.num_imputer.transform(X[self.numeric_cols_])

        if len(self.categorical_cols_) > 0:
            X[self.categorical_cols_] = self.cat_imputer.transform(X[self.categorical_cols_])

        return X

# **Scaling**

In [31]:
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class NumericScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        self.numeric_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.scaler.fit(X[self.numeric_cols_])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.numeric_cols_] = self.scaler.transform(X[self.numeric_cols_])
        return X

# **One-Hot-Encoding**

In [32]:
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class CategoricalOHE(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None, max_categories=50):
        self.cols = cols
        self.max_categories = max_categories
        self.encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            max_categories=max_categories
        )

    def fit(self, X, y=None):
        X = X.copy()

        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object"]).columns.tolist()
        else:
            self.cols_ = self.cols

        X_cat = X[self.cols_].fillna("missing")
        self.encoder.fit(X_cat)

        return self

    def transform(self, X):
        X = X.copy()

        X_cat = X[self.cols_].fillna("missing")
        X_ohe = self.encoder.transform(X_cat)

        X_num = X.drop(columns=self.cols_)

        # convert numeric part to sparse too
        X_num_sparse = sparse.csr_matrix(X_num.to_numpy())

        return sparse.hstack([X_num_sparse, X_ohe]).tocsr()

# **Feature Engineering**

In [33]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, uid_cols=("card1", "addr1")):
        self.uid_cols = uid_cols
        self.uid_means_ = None

    def fit(self, X, y=None):
        X = X.copy()

        uid = self._make_uid(X)

        self.uid_means_ = (
            pd.DataFrame({
                "uid": uid,
                "TransactionAmt": X["TransactionAmt"]
            })
            .groupby("uid")["TransactionAmt"]
            .mean()
        )

        return self

    def transform(self, X):
        X = X.copy()

        X["hour"] = (X["TransactionDT"] // 3600) % 24

        id_cols = [col for col in X.columns if col.startswith("id_")]
        if id_cols:
            X["identity_missing"] = X[id_cols].isna().sum(axis=1)
        else:
            X["identity_missing"] = 0

        uid = self._make_uid(X)

        uid_mean = uid.map(self.uid_means_)

        global_mean = self.uid_means_.mean()
        uid_mean = uid_mean.fillna(global_mean)

        X["uid_amt_mean"] = uid_mean

        X["uid_amt_diff"] = X["TransactionAmt"] - X["uid_amt_mean"]

        return X

    def _make_uid(self, X):
        uid = X[self.uid_cols[0]].astype(str)
        for col in self.uid_cols[1:]:
            uid += "_" + X[col].astype(str)
        return uid

# **Feature Selection**

In [34]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold):
        self.threshold = threshold
        self.cols_to_drop_ = None

    def fit(self, X, y=None):
        X = X.copy()

        numeric_cols = X.select_dtypes(include=[np.number]).columns
        X_num = X[numeric_cols]

        corr_matrix = X_num.corr().abs()

        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        self.cols_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]

        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [35]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class UselessFeatureDropper(BaseEstimator, TransformerMixin):
    def __init__(self,  variance_threshold=0.0):
        self.variance_threshold = variance_threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        X = X.copy()
        self.cols_to_drop_ = []

        constant_cols = [
            col for col in X.columns
            if X[col].nunique(dropna=False) <= 1
        ]

        numeric_cols = X.select_dtypes(include=["number"]).columns

        near_zero_var_cols = [
            col for col in numeric_cols
            if X[col].var(skipna=True) <= self.variance_threshold
        ]

        self.cols_to_drop_ = list(set(
            constant_cols + near_zero_var_cols
        ))

        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [36]:
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

mlflow.set_experiment("logistic_regression_training")

train_trx, val_trx = train_test_split(
    transaction,
    test_size=0.2,
    random_state=42,
    stratify=transaction["isFraud"]
)



y_train = train_trx["isFraud"]
y_val = val_trx["isFraud"]
train_trx = train_trx.drop(columns=["isFraud"])
val_trx = val_trx.drop(columns=["isFraud"])

X_train_raw = {
    "transaction": train_trx,
    "identity": identity
}

X_val_raw = {
    "transaction": val_trx,
    "identity": identity
}

cleaning_fe_pipeline = Pipeline([
    ("transaction_cleaner", TransactionCleaner()),
    ("drop_useless", UselessFeatureDropper()),
    ("features", FraudFeatureEngineer()),
    ("correlation_filter", CorrelationFilter(threshold=0.95)),
    ("imputer", LogisticImputer()),
    ("scaler", NumericScaler()),
    ("cat_ohe", CategoricalOHE())
])

with mlflow.start_run(run_name="cleaning_feature_engineering"):

    X_train_processed = cleaning_fe_pipeline.fit_transform(X_train_raw, y_train)
    X_val_processed = cleaning_fe_pipeline.transform(X_val_raw)

    # parameters
    mlflow.log_param("transaction_cleaner", True)
    mlflow.log_param("drop_useless", True)
    mlflow.log_param("feature_engineering", True)
    mlflow.log_param("correlation_filter", True)
    mlflow.log_param("correlation_threshold", 0.95)
    mlflow.log_param("imputer", "enabled")
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_param("categorical_encoder", "OneHotEncoder")
    mlflow.log_param("max_categories", 50)

    # outcomes
    mlflow.log_param("train_rows_before", train_trx.shape[0])
    mlflow.log_param("train_transaction_cols_before", train_trx.shape[1])
    mlflow.log_param("val_rows_before", val_trx.shape[0])
    mlflow.log_param("val_transaction_cols_before", val_trx.shape[1])
    mlflow.log_param("identity_cols_before", identity.shape[1])

    mlflow.log_param("train_rows_after", X_train_processed.shape[0])
    mlflow.log_param("train_cols_after", X_train_processed.shape[1])
    mlflow.log_param("val_rows_after", X_val_processed.shape[0])
    mlflow.log_param("val_cols_after", X_val_processed.shape[1])

    dropped_cols = cleaning_fe_pipeline.named_steps["drop_useless"].cols_to_drop_
    corr_dropped_cols = cleaning_fe_pipeline.named_steps["correlation_filter"].cols_to_drop_

    mlflow.log_param("num_useless_dropped_cols", len(dropped_cols))
    mlflow.log_param("num_corr_dropped_cols", len(corr_dropped_cols))

🏃 View run cleaning_feature_engineering at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/2/runs/a4542bc4bc12476dbaf55eba51436543
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/2


# **Logistic Regression**

In [37]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    recall_score, f1_score, precision_score, log_loss
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import pandas as pd
import mlflow

mlflow.set_experiment("logistic_regression_training")

param_list = [
    {"C": 0.1, "penalty": "l1"},
    {"C": 1.0, "penalty": "l1"},
    {"C": 0.1, "penalty": "l2"},
    {"C": 1.0, "penalty": "l2"},
]

results = []

for params in param_list:
    with mlflow.start_run(run_name="LogisticRegression_training"):

        model = LogisticRegression(
            **params,
            solver="saga",
            class_weight="balanced",
            max_iter=3000,
            n_jobs=2,
            tol=1e-3,
            random_state=42
        )

        temp_pipeline = Pipeline([
            ("transaction_cleaner", TransactionCleaner()),
            ("drop_useless", UselessFeatureDropper()),
            ("features", FraudFeatureEngineer()),
            ("correlation_filter", CorrelationFilter(threshold=0.95)),
            ("imputer", LogisticImputer()),
            ("scaler", NumericScaler()),
            ("cat_ohe", CategoricalOHE()),
            ("model", model)
        ])

        temp_pipeline.fit(X_train_raw, y_train)

        train_preds = temp_pipeline.predict_proba(X_train_raw)[:, 1]
        val_preds = temp_pipeline.predict_proba(X_val_raw)[:, 1]

        train_auc = roc_auc_score(y_train, train_preds)
        val_auc = roc_auc_score(y_val, val_preds)
        auc_gap = train_auc - val_auc

        y_val_pred = (val_preds >= 0.5).astype(int)

        pr_auc = average_precision_score(y_val, val_preds)
        precision = precision_score(y_val, y_val_pred, zero_division=0)
        recall = recall_score(y_val, y_val_pred, zero_division=0)
        f1 = f1_score(y_val, y_val_pred, zero_division=0)
        ll = log_loss(y_val, val_preds)

        mlflow.log_params(params)

        mlflow.log_metric("train_roc_auc", train_auc)
        mlflow.log_metric("val_roc_auc", val_auc)
        mlflow.log_metric("auc_gap", auc_gap)
        mlflow.log_metric("val_pr_auc", pr_auc)
        mlflow.log_metric("val_precision", precision)
        mlflow.log_metric("val_recall", recall)
        mlflow.log_metric("val_f1", f1)
        mlflow.log_metric("val_log_loss", ll)

        results.append({
            **params,
            "train_roc_auc": train_auc,
            "val_roc_auc": val_auc,
            "auc_gap": auc_gap,
            "val_pr_auc": pr_auc,
            "val_precision": precision,
            "val_recall": recall,
            "val_f1": f1,
            "val_log_loss": ll
        })

        print(params)
        print("Train AUC:", train_auc)
        print("Val AUC:", val_auc)
        print("Gap:", auc_gap)
        print("-" * 40)

results_df = pd.DataFrame(results).sort_values("val_roc_auc", ascending=False)
results_df

{'C': 0.1, 'penalty': 'l1'}
Train AUC: 0.8693676480598536
Val AUC: 0.8678464503386971
Gap: 0.001521197721156442
----------------------------------------
🏃 View run LogisticRegression_training at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/2/runs/32d09d5c12e546ed8a64580eed39b328
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/2
{'C': 1.0, 'penalty': 'l1'}
Train AUC: 0.8695578450578325
Val AUC: 0.8679983295074654
Gap: 0.001559515550367152
----------------------------------------
🏃 View run LogisticRegression_training at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/2/runs/10e233dd5cbf4478be6468412c743fbd
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/2
{'C': 0.1, 'penalty': 'l2'}
Train AUC: 0.869544656999456
Val AUC: 0.8679818347470196
Gap: 0.0015628222524364244
----------------------------------------
🏃 View run LogisticRegression_training at: https://dagshub.com/mkhak23/

,C,penalty,train_roc_auc,val_roc_auc,auc_gap,val_pr_auc,val_precision,val_recall,val_f1,val_log_loss
3,1.0,l2,0.869573,0.868009,0.001564,0.434740,0.138386,0.741350,0.233234,0.425880
1,1.0,l1,0.869558,0.867998,0.001560,0.434730,0.138320,0.740866,0.233118,0.425901
2,0.1,l2,0.869545,0.867982,0.001563,0.434687,0.138283,0.740866,0.233064,0.425927
0,0.1,l1,0.869368,0.867846,0.001521,0.434456,0.138124,0.740382,0.232815,0.426173


In [38]:
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

best_params = results_df.iloc[0][["C", "penalty"]].to_dict()
best_params["C"] = float(best_params["C"])

final_model = LogisticRegression(
    **best_params,
    solver="saga",
    class_weight="balanced",
    max_iter=3000,
    n_jobs=2,
    tol=1e-3,
    random_state=42
)

final_pipeline = Pipeline([
    ("transaction_cleaner", TransactionCleaner()),
    ("drop_useless", UselessFeatureDropper()),
    ("features", FraudFeatureEngineer()),
    ("correlation_filter", CorrelationFilter(threshold=0.95)),
    ("imputer", LogisticImputer()),
    ("scaler", NumericScaler()),
    ("cat_ohe", CategoricalOHE()),
    ("model", final_model)
])

X_full_raw = {
    "transaction": transaction.drop(columns=["isFraud"]),
    "identity": identity
}

y_full = transaction["isFraud"]

final_pipeline.fit(X_full_raw, y_full)

mlflow.set_experiment("logistic_regression_training")

with mlflow.start_run(run_name="final_best_logistic_regression"):
    mlflow.log_params(best_params)
    mlflow.log_param("solver", "saga")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 3000)
    mlflow.log_param("tol", 1e-3)
    mlflow.log_param("correlation_threshold", 0.95)

    mlflow.log_metric("best_val_roc_auc", results_df.iloc[0]["val_roc_auc"])
    mlflow.log_metric("best_val_pr_auc", results_df.iloc[0]["val_pr_auc"])
    mlflow.log_metric("best_val_f1", results_df.iloc[0]["val_f1"])

    mlflow.sklearn.log_model(
        final_pipeline,
        name="final_pipeline"
    )

2026/05/05 21:57:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run final_best_logistic_regression at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/2/runs/dce24fbdb4ec423a99927523a5e343a0
🧪 View experiment at: https://dagshub.com/mkhak23/ML_assignment2.mlflow/#/experiments/2
